## Report
```
The input DNA string (up to 1000 nt) is converted to an integer array (A=0, C=1, G=2, T=3) on the host and transferred to the GPU.
A CUDA kernel assigns one thread per nucleotide and each thread performs a global atomicAdd on one of four counters corresponding to its base.
The final counts for the input are printed.
```

In [1]:
%%writefile counting_dna_nucleotides.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <cuda_runtime.h>

#define DNA_INPUT "AGCTTTTCATTCTGACTGCAACGGGCAATATGTCTCTGTGTGGATTAAAAAAAGAGTGTCTGATAGCAGC"

__global__ void counting_dna_nucleotides_kernel(const int *seq, unsigned long long *counters, int n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < n) {
        atomicAdd(&counters[seq[idx]], 1ULL);
    }
}

static void encode_dna(const char *s, int *out, int n)
{
    for (int i = 0; i < n; i++) {
        switch (s[i]) {
            case 'A':
                out[i] = 0;

                break;

            case 'C':
                out[i] = 1;

                break;

            case 'G':
                out[i] = 2;

                break;

            default:
                out[i] = 3;

                break;
        }
    }
}

void launch_dna_kernel(const int *h_seq, int n)
{
    int *d_seq = NULL;
    unsigned long long *d_counters = NULL;

    cudaMalloc((void **)&d_seq, n * sizeof(int));
    cudaMalloc((void **)&d_counters, 4 * sizeof(unsigned long long));

    cudaMemset(d_counters, 0, 4 * sizeof(unsigned long long));
    cudaMemcpy(d_seq, h_seq, n * sizeof(int), cudaMemcpyHostToDevice);

    int block_size = 256;
    int grid_size = (n + block_size - 1) / block_size;

    counting_dna_nucleotides_kernel<<<grid_size, block_size>>>(d_seq, d_counters, n);

    cudaDeviceSynchronize();

    unsigned long long h_counters[4] = {0};

    cudaMemcpy(h_counters, d_counters, 4 * sizeof(unsigned long long), cudaMemcpyDeviceToHost);

    printf("%llu %llu %llu %llu\n", h_counters[0], h_counters[1], h_counters[2], h_counters[3]);

    cudaFree(d_seq);
    cudaFree(d_counters);
}

int main(void)
{
    const char *dna = DNA_INPUT;
    int n = (int)strlen(dna);
    int *h_seq = (int *)malloc(n * sizeof(int));

    encode_dna(dna, h_seq, n);

    launch_dna_kernel(h_seq, n);

    free(h_seq);

    return 0;
}

Writing counting_dna_nucleotides.cu


In [2]:
!nvcc -arch=sm_75 counting_dna_nucleotides.cu -o counting_dna_nucleotides

In [3]:
!./counting_dna_nucleotides

20 12 17 21
